# 📐 RAGAS II — A Comprehensive Guide to RAG Evaluation Metrics

Based on the article: *[RAGAS for RAG in LLMs: A Comprehensive Guide to Evaluation Metrics](https://dkaarthick.medium.com/ragas-for-rag-in-llms-a-comprehensive-guide-to-evaluation-metrics-3aca142d6e38)*

---

## Why do we need RAGAS?

Traditional text generation metrics like **BLEU** and **ROUGE** were designed for tasks like machine translation and summarisation. They measure *surface-level similarity* — how many n-grams overlap between a generated text and a reference text.

For **RAG (Retrieval-Augmented Generation)** systems, these metrics fall short:

| Problem | Why BLEU/ROUGE fails |
|---------|---------------------|
| Factual accuracy | A response can have high n-gram overlap but contain hallucinations |
| Context grounding | They don't check whether the answer is *supported* by retrieved docs |
| Retrieval quality | They ignore whether the right documents were retrieved at all |
| Relevance | They don't penalise verbose answers that drift off-topic |

**RAGAS** (Retrieval Augmented Generation Assessment) fixes this with a suite of metrics purpose-built for the RAG evaluation problem.

---

## The RAGAS Metric Suite

| Metric | What it measures | Inputs required |
|--------|------------------|-----------------|
| **Faithfulness** | Is the answer factually supported by the retrieved context? | answer + context |
| **Answer Relevancy** | Does the answer actually address the question? | question + answer |
| **Answer Correctness** | How accurate is the answer compared to ground truth? | answer + ground truth |
| **Context Precision** | Are retrieved chunks relevant, or is there noise? | question + context + ground truth |
| **Context Recall** | Did the retrieval capture all necessary information? | context + ground truth |

Together these metrics give you a **360° view** of where your RAG pipeline is failing: the LLM, the retriever, or both.

## 📦 1. Install Dependencies

RAGAS uses an LLM internally to compute some metrics (e.g. faithfulness decomposes the answer into claims and checks each one against the context). By default it uses **OpenAI**.

> **Using Ollama instead?** See the optional cell near the bottom of this notebook for a drop-in LLM swap that works fully offline.

In [ ]:
!pip install -q ragas datasets openai
print("✅ Dependencies ready")

## 🔑 2. Configure Your LLM

RAGAS needs an LLM to evaluate metrics that involve language understanding (faithfulness, answer relevancy, answer correctness). Set your OpenAI API key below.

> If you're running on **Google Colab**, use `userdata.get('OPENAI_API_KEY')` instead of `os.getenv`.

In [ ]:
import os

# Option A: set directly (not recommended for shared notebooks)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option B: read from environment (recommended)
api_key = os.getenv("OPENAI_API_KEY")

# Option C: Colab secrets
# from google.colab import userdata
# api_key = userdata.get('OPENAI_API_KEY')

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key
    print("✅ OpenAI API key loaded")
else:
    print("⚠️  No OPENAI_API_KEY found — set it or use the Ollama alternative below")

## 📥 3. Import the Libraries

We import:
- `Dataset` from Hugging Face — RAGAS expects data in this format
- `evaluate` — the main RAGAS evaluation function
- All five metrics we want to measure

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall,
)

print("✅ Imports successful")

## 🗃️ 4. Build the Evaluation Dataset

RAGAS expects a dataset with **four fields**:

| Field | Description | Type |
|-------|-------------|------|
| `question` | The user's input query | `str` |
| `answer` | The response generated by your RAG model | `str` |
| `contexts` | The documents retrieved to support the answer | `List[str]` |
| `ground_truth` | The known-correct reference answer | `str` |

> **Note:** Older RAGAS versions used `query`, `generated_response`, and `retrieved_documents` as field names. Newer versions (≥0.1.x) standardised on `question`, `answer`, `contexts`, and `ground_truth`. Check your version if you hit field errors.

In a real project, you would build this dataset by:
1. Running your RAG pipeline on a test set of questions
2. Collecting the generated answers and retrieved documents
3. Pairing them with ground truth answers (written by a human or derived from a curated QA set)

In [ ]:
# Example evaluation data — one question about Paris
# In production you'd have dozens or hundreds of rows
data = {
    # The input question sent to the RAG model
    "question": [
        "What is the capital of France?"
    ],

    # The answer your RAG model generated
    "answer": [
        "Paris is the capital of France."
    ],

    # The document chunks retrieved by the retriever
    # Each entry is a LIST of strings (multiple chunks per question)
    "contexts": [
        [
            "Paris is the capital of France. "
            "It is a major European city known for its culture, art, and cuisine."
        ]
    ],

    # The reference answer (ground truth) — used by answer_correctness,
    # context_precision, and context_recall
    "ground_truth": [
        "Paris"
    ],
}

print("Dataset fields:", list(data.keys()))
print(f"Number of rows: {len(data['question'])}")

## 🔄 5. Convert to a Hugging Face Dataset

RAGAS uses the Hugging Face `Dataset` class as its input format. Converting is one line — it validates field names and types at this stage, so any mismatches surface early.

In [ ]:
dataset = Dataset.from_dict(data)

print("Dataset schema:")
print(dataset)
print("\nFirst row preview:")
print(dataset[0])

## 📏 6. Define the Metrics

Let's understand each metric before we run them:

---

### 1. Faithfulness
Checks whether every factual claim in the *answer* is supported by the *context*. RAGAS decomposes the answer into atomic claims and verifies each against the retrieved documents using the LLM.

- **Score range**: 0–1 (higher = more grounded in context, fewer hallucinations)
- **Low score means**: the model is making things up beyond what the context supports

---

### 2. Answer Relevancy
Measures how directly the answer responds to the question. It generates reverse questions from the answer and checks alignment with the original question.

- **Score range**: 0–1 (higher = answer stays on topic)
- **Low score means**: the answer is verbose, drifting, or answering a different question

---

### 3. Answer Correctness
Compares the generated answer against the ground truth. It combines factual overlap (F1 over facts) with semantic similarity.

- **Score range**: 0–1 (higher = closer to ground truth)
- **Low score means**: the answer is factually wrong or incomplete vs. the known correct answer

---

### 4. Context Precision
Are the retrieved documents actually useful? Measures what fraction of the retrieved context is relevant to answering the question (signal-to-noise ratio of the retriever).

- **Score range**: 0–1 (higher = retriever returns precise, relevant chunks)
- **Low score means**: retriever is pulling in noisy or off-topic documents

---

### 5. Context Recall
Did the retriever find *all* the information needed? Checks whether the ground truth answer can be attributed to the retrieved context.

- **Score range**: 0–1 (higher = retriever captured everything needed)
- **Low score means**: the retriever missed important documents — the model can't answer well with what it was given

In [ ]:
# Bundle all five metrics into a list
# You can comment out metrics you don't need to speed up evaluation
metrics = [
    faithfulness,        # answer grounded in context?
    answer_relevancy,    # answer on-topic?
    answer_correctness,  # answer factually correct vs ground truth?
    context_precision,   # retrieved docs precise?
    context_recall,      # retrieved docs complete?
]

print(f"Evaluating {len(metrics)} metrics:")
for m in metrics:
    print(f"  - {m.name}")

## 🚀 7. Run the Evaluation

The `evaluate()` call orchestrates everything:
1. It sends LLM prompts to compute each metric
2. Returns a `Result` object with per-metric average scores

> ⏳ This will make several OpenAI API calls. For 1 row with 5 metrics, expect ~10–30 seconds.

In [ ]:
print("Running RAGAS evaluation...")
results = evaluate(dataset, metrics=metrics)
print("✅ Evaluation complete")

## 📊 8. Inspect the Results

RAGAS returns a `Result` object that behaves like a dictionary. Let's look at the scores two ways:
- A simple printed summary
- A Pandas DataFrame for easy sorting and filtering at scale

In [ ]:
# Simple print — matches the article output format
print("=" * 40)
print("RAGAS Evaluation Results")
print("=" * 40)
for metric_name, score in results.items():
    bar = "█" * int(score * 20)  # ASCII progress bar
    print(f"{metric_name:<22} {score:.2f}  {bar}")
print("=" * 40)

In [ ]:
# DataFrame view — useful when you have many rows
import pandas as pd

df = results.to_pandas()
print("Per-row results (useful for multi-question datasets):")
df

## 🦙 Optional: Use Ollama Instead of OpenAI

If you want to run RAGAS **fully offline** without any OpenAI API key, you can swap in an Ollama model. Install Ollama, pull a model, and configure RAGAS before calling `evaluate()`:

```bash
# Terminal setup
ollama serve
ollama pull llama3.2
```

Then in the notebook, replace the `evaluate()` call with:

In [ ]:
# ── OPTIONAL: Ollama backend ──────────────────────────────────────────────────
# Uncomment this cell and run it INSTEAD of the standard evaluate() cell above.
# Requires: pip install langchain-ollama && ollama pull llama3.2
#
# from langchain_ollama import ChatOllama
# from ragas.llms import LangchainLLMWrapper
#
# ollama_llm = LangchainLLMWrapper(ChatOllama(model="llama3.2", base_url="http://localhost:11434"))
#
# # Inject the custom LLM into each metric
# for metric in metrics:
#     metric.llm = ollama_llm
#
# results = evaluate(dataset, metrics=metrics)
# print("✅ Ollama evaluation complete")

print("(Cell is commented out — uncomment to use Ollama)")

## 🔁 9. A Multi-Question Example

Real evaluations need more than one question to be meaningful. Here's the same pipeline with three diverse questions so you can see how scores vary across different query types.

In [ ]:
# A richer dataset with 3 questions to better illustrate metric variation
multi_data = {
    "question": [
        "What is the capital of France?",
        "Who invented the telephone?",
        "What is the boiling point of water?",
    ],
    "answer": [
        # Good answer — faithful and correct
        "Paris is the capital of France.",
        # Slightly misleading — Elisha Gray also filed a patent the same day
        "Alexander Graham Bell invented the telephone in 1876.",
        # Wrong unit mentioned — tests answer_correctness
        "Water boils at 100 degrees Celsius at sea level.",
    ],
    "contexts": [
        ["Paris is the capital and most populous city of France."],
        [
            "Alexander Graham Bell was awarded the first patent for the telephone in 1876. "
            "Elisha Gray also independently developed a telephone design around the same time."
        ],
        [
            "The boiling point of water is 100°C (212°F) at standard atmospheric pressure (1 atm). "
            "At higher altitudes, where atmospheric pressure is lower, water boils at a lower temperature."
        ],
    ],
    "ground_truth": [
        "Paris",
        "Alexander Graham Bell invented the telephone.",
        "100 degrees Celsius (212 degrees Fahrenheit) at sea level.",
    ],
}

multi_dataset = Dataset.from_dict(multi_data)
print(f"Multi-question dataset: {len(multi_dataset)} rows")
multi_dataset.to_pandas()[["question", "answer"]]

In [ ]:
# Evaluate the multi-question dataset
# (Costs more API tokens — comment out if budget is a concern)
print("Running multi-question evaluation...")
multi_results = evaluate(multi_dataset, metrics=metrics)

print("\nAggregate scores:")
for metric_name, score in multi_results.items():
    print(f"  {metric_name:<22} {score:.3f}")

print("\nPer-row breakdown:")
multi_results.to_pandas()

## 📌 10. Interpreting Results & What to Fix

Use this table as a diagnostic guide when you see low scores:

| Low score on... | Likely root cause | What to try |
|----------------|-------------------|-------------|
| **Faithfulness** | LLM is hallucinating | Add stricter prompting ("only use the provided context"); use a smaller, less creative model |
| **Answer Relevancy** | LLM is verbose or off-topic | Tune your prompt template; add output length constraints |
| **Answer Correctness** | LLM gets facts wrong | Check context quality first; consider a more capable model |
| **Context Precision** | Retriever returns noise | Reduce `top_k`; improve your embedding model; use a reranker |
| **Context Recall** | Retriever misses relevant docs | Increase `top_k`; improve chunking strategy; try hybrid search |

---

## 🧾 Conclusion

RAGAS provides a **purpose-built evaluation framework** for RAG systems that goes far beyond BLEU/ROUGE:

- **BLEU/ROUGE** measure *lexical overlap* — useful for MT and summarisation, but blind to factual accuracy and retrieval quality
- **RAGAS metrics** capture the full RAG pipeline: retrieval quality (precision/recall) AND generation quality (faithfulness, relevancy, correctness)

Running RAGAS regularly on a gold-standard test set gives you a **regression safety net** — if you change your retriever, embeddings, or LLM, you'll immediately see which metrics improved or degraded.

**Next steps:**
- Scale up: build a 50–100 question evaluation set from your real use case
- Automate: run RAGAS in CI on every model/retriever change
- Combine: pair RAGAS with human spot-checks for high-stakes applications
- Explore: `ragas` also supports custom metrics and domain-specific evaluation — check the [official docs](https://docs.ragas.io)